In [ ]:
from google.cloud import bigquery
import pandas as pd
import numpy as np

In [6]:
client = bigquery.Client(project="pacey32-agency")

In [25]:
sql = """
WITH profile AS (
    SELECT
        playerId,
        ANY_VALUE(player_name) AS player,
        ANY_VALUE(age) AS age,
        ANY_VALUE(position) AS position,
        ANY_VALUE(heightInCentimeters) AS height_cm,
        ANY_VALUE(weightInKilograms) AS weight_kg,
        ANY_VALUE(draftRound) AS draftRound,
        ANY_VALUE(draftOverall) AS draftPick,
        ANY_VALUE(draftYear) AS draftYear,
        ANY_VALUE(draftTeam) AS draftTeam,
        ANY_VALUE(shoots_catches) AS shoots_catches,
        ANY_VALUE(birth_country) AS birthCountry
    FROM `pacey32-agency.Comparison.01_PlayerProfile`
    WHERE activeFlag = 1
    GROUP BY playerId
),

stats AS (
    SELECT
        playerId,
        season,
        SUM(games_played) AS games,
        SUM(goals) AS goals,
        SUM(assists) AS assists,
        SUM(points) AS points,
        SAFE_DIVIDE(SUM(goals), SUM(games_played)) AS goals_per_game,
        SAFE_DIVIDE(SUM(assists), SUM(games_played)) AS assists_per_game,
        SAFE_DIVIDE(SUM(points), SUM(games_played)) AS points_per_game,
        SAFE_DIVIDE(SUM(toi_minutes), SUM(games_played)) AS avg_toi_minutes,
        SAFE_DIVIDE(SUM(goals_per_60 * toi_minutes), SUM(toi_minutes)) AS goals_per_60,
        SAFE_DIVIDE(SUM(assists_per_60 * toi_minutes), SUM(toi_minutes)) AS assists_per_60,
        SAFE_DIVIDE(SUM(points_per_60 * toi_minutes), SUM(toi_minutes)) AS points_per_60
    FROM `pacey32-agency.Comparison.03_PlayerSeasonStats`
    WHERE seasonPart = 'RegularSeason'
    GROUP BY playerId, season
),

latest AS (
    SELECT *
    FROM stats
    WHERE games >= 40
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY playerId
        ORDER BY season DESC
    ) = 1
)

SELECT
    p.*,
    s.season,
    s.games,
    s.goals,
    s.assists,
    s.points,
    s.goals_per_game,
    s.assists_per_game,
    s.points_per_game,
    s.avg_toi_minutes,
    s.goals_per_60,
    s.assists_per_60,
    s.points_per_60
FROM profile p
LEFT JOIN latest s
    ON p.playerId = CAST(s.playerId AS INT64)
"""

df_players = client.query(sql).to_dataframe()

In [26]:
print(f"Players: {len(df_players):,}")
display(df_players.head())

Players: 1,076


,playerId,player,age,position,height_cm,weight_kg,draftRound,draftPick,draftYear,draftTeam,...,goals,assists,points,goals_per_game,assists_per_game,points_per_game,avg_toi_minutes,goals_per_60,assists_per_60,points_per_60
0,8474141,Patrick Kane,38,R,178,80,1,1,2007,CHI,...,16,41,57,0.238806,0.611940,0.850746,17.695522,0.81,2.07,2.88
1,8476467,Jamie Oleksiak,34,D,201,114,1,14,2011,DAL,...,6,10,16,0.076923,0.128205,0.205128,16.926923,0.27,0.45,0.73
2,8476981,Josh Anderson,32,R,191,103,4,95,2012,CBJ,...,14,9,23,0.194444,0.125000,0.319444,14.022222,0.83,0.53,1.37
3,8477034,Jaycob Megna,34,D,198,97,7,210,2012,ANA,...,0,2,2,0.000000,0.045455,0.045455,17.522727,0.00,0.16,0.16
4,8477463,Steven Santini,31,D,191,98,2,42,2013,NJD,...,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
features = [
    "age",
    "height_cm",
    "weight_kg",
    "goals_per_game",
    "assists_per_game",
    "points_per_game",
    "avg_toi_minutes",
    "goals_per_60",
    "assists_per_60",
    "points_per_60"
]

df_model = df_players.dropna(subset=features).copy()

print(f"Players available for comparison: {len(df_model):,}")

Players available for comparison: 762


In [28]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X = scaler.fit_transform(df_model[features])

print(X.shape)

(762, 10)


In [29]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(X)

print(similarity_matrix.shape)

(762, 762)


In [30]:
def find_comparables(player_id, n=10):
    idx = df_model.index[df_model["playerId"] == player_id][0]
    pos = df_model.index.get_loc(idx)

    target_position = df_model.loc[idx, "position"]
    scores = similarity_matrix[pos]

    results = df_model[[
        "playerId", "player", "age", "position",
        "games", "goals", "assists", "points",
        "points_per_game", "avg_toi_minutes",
        "height_cm", "weight_kg"
    ]].copy()

    results["similarity"] = scores

    results = results[
        (results["playerId"] != player_id) &
        (results["position"] == target_position)
    ]

    results = results.sort_values("similarity", ascending=False).head(n)
    results["similarity"] = (results["similarity"] * 100).round(1)
    return results.reset_index(drop=True)


In [31]:
leo_comps = find_comparables(8484153, 10)
display(leo_comps)

,playerId,player,age,position,games,goals,assists,points,points_per_game,avg_toi_minutes,height_cm,weight_kg,similarity
0,8481528,Dylan Cozens,25,C,82,29,31,60,0.731707,17.040244,191,93,96.8
1,8484166,Adam Fantilli,22,C,82,24,35,59,0.719512,18.902439,188,93,95.8
2,8480014,Gabriel Vilardi,27,C,82,30,39,69,0.841463,18.740244,191,98,95.5
3,8484801,Macklin Celebrini,20,C,82,45,70,115,1.402439,21.315854,183,86,94.0
4,8481596,Shane Pinto,26,C,72,23,24,47,0.652778,18.712500,191,93,93.3
5,8480039,Martin Necas,27,C,78,38,62,100,1.282051,21.497436,191,88,93.0
6,8482113,Anton Lundell,25,C,64,18,27,45,0.703125,19.151563,185,89,92.6
7,8480002,Nico Hischier,27,C,82,28,38,66,0.804878,20.717073,185,91,92.6
8,8480064,Josh Norris,27,C,44,13,21,34,0.772727,15.831818,188,89,91.9
9,8478401,Pavel Zacha,29,C,78,30,37,67,0.858974,16.846154,193,96,91.5


In [33]:
am_comps = find_comparables(8479318, 10)
display(am_comps)

,playerId,player,age,position,games,goals,assists,points,points_per_game,avg_toi_minutes,height_cm,weight_kg,similarity
0,8479420,Tage Thompson,29,C,81,40,41,81,1.000000,19.243210,198,100,97.1
1,8480014,Gabriel Vilardi,27,C,82,30,39,69,0.841463,18.740244,191,98,95.9
2,8479987,Morgan Geekie,28,C,81,41,29,70,0.864198,17.406173,191,96,95.9
3,8478401,Pavel Zacha,29,C,78,30,37,67,0.858974,16.846154,193,96,94.0
4,8477500,Bo Horvat,31,C,68,31,27,58,0.852941,20.763235,185,102,94.0
5,8477946,Dylan Larkin,30,C,74,34,33,67,0.905405,20.186486,185,93,93.9
6,8476459,Mika Zibanejad,33,C,81,35,45,80,0.987654,20.897531,188,94,92.4
7,8481596,Shane Pinto,26,C,72,23,24,47,0.652778,18.712500,191,93,92.1
8,8481528,Dylan Cozens,25,C,82,29,31,60,0.731707,17.040244,191,93,90.5
9,8478493,Joel Eriksson Ek,29,C,70,19,32,51,0.728571,19.078571,191,94,90.3


# Model v2

In [ ]:
features = [
    "age",
    "height_cm",
    "weight_kg",
    "draftPick",
    "goals_per_game",
    "assists_per_game",
    "points_per_game",
    "avg_toi_minutes",
    "goals_per_60",
    "assists_per_60",
    "points_per_60"
]

df_model = df_players.dropna(subset=features).copy()

print(f"Players available for comparison: {len(df_model):,}")

In [34]:
from sklearn.preprocessing import StandardScaler

scaler_v2 = StandardScaler()

X_v2 = scaler_v2.fit_transform(df_model[features])

print(X_v2.shape)

(762, 10)


In [35]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix_v2 = cosine_similarity(X_v2)

print(similarity_matrix_v2.shape)

(762, 762)


In [44]:
def find_comparables_v2(player_id, n=10):
    idx = df_model.index[df_model["playerId"] == player_id][0]
    pos = df_model.index.get_loc(idx)

    target_position = df_model.loc[idx, "position"]
    target_hand = df_model.loc[idx, "shoots_catches"]

    results = df_model[[
        "playerId", "player", "age", "position", "shoots_catches",
        "draftPick", "games", "goals", "assists", "points",
        "goals_per_game", "assists_per_game", "points_per_game",
        "avg_toi_minutes", "goals_per_60", "assists_per_60", "points_per_60",
        "height_cm", "weight_kg"
    ]].copy()

    results["base_similarity"] = similarity_matrix_v2[pos]
    results["same_hand"] = (results["shoots_catches"] == target_hand).astype(int)

    hand_weight = 0.05 if target_position == "D" else 0.02

    results["similarity"] = (
        results["base_similarity"] * (1 - hand_weight) +
        results["same_hand"] * hand_weight
    )

    results = results[
        (results["playerId"] != player_id) &
        (results["position"] == target_position)
    ]

    results = results.sort_values("similarity", ascending=False).head(n)

    results["base_similarity"] = (results["base_similarity"] * 100).round(1)
    results["similarity"] = (results["similarity"] * 100).round(1)

    return results.reset_index(drop=True)

In [45]:
display(find_comparables_v2(8484153, 10))

,playerId,player,age,position,shoots_catches,draftPick,games,goals,assists,points,...,points_per_game,avg_toi_minutes,goals_per_60,assists_per_60,points_per_60,height_cm,weight_kg,base_similarity,same_hand,similarity
0,8484166,Adam Fantilli,22,C,L,3,82,24,35,59,...,0.719512,18.902439,0.93,1.35,2.28,188,93,95.8,1,95.9
1,8481528,Dylan Cozens,25,C,R,7,82,29,31,60,...,0.731707,17.040244,1.25,1.33,2.58,191,93,96.8,0,94.8
2,8484801,Macklin Celebrini,20,C,L,1,82,45,70,115,...,1.402439,21.315854,1.54,2.40,3.95,183,86,94.0,1,94.1
3,8480014,Gabriel Vilardi,27,C,R,11,82,30,39,69,...,0.841463,18.740244,1.17,1.52,2.69,191,98,95.5,0,93.6
4,8482113,Anton Lundell,25,C,L,12,64,18,27,45,...,0.703125,19.151563,0.88,1.32,2.20,185,89,92.6,1,92.8
5,8480002,Nico Hischier,27,C,L,1,82,28,38,66,...,0.804878,20.717073,0.99,1.34,2.33,185,91,92.6,1,92.7
6,8480064,Josh Norris,27,C,L,19,44,13,21,34,...,0.772727,15.831818,1.12,1.81,2.93,188,89,91.9,1,92.1
7,8478401,Pavel Zacha,29,C,L,6,78,30,37,67,...,0.858974,16.846154,1.37,1.69,3.06,193,96,91.5,1,91.7
8,8482116,Tim Stützle,24,C,L,3,80,36,51,87,...,1.087500,20.267500,1.33,1.89,3.22,183,85,91.4,1,91.6
9,8481596,Shane Pinto,26,C,R,32,72,23,24,47,...,0.652778,18.712500,1.02,1.07,2.09,191,93,93.3,0,91.4


In [46]:
display(find_comparables_v2(8479318, 10))

,playerId,player,age,position,shoots_catches,draftPick,games,goals,assists,points,...,points_per_game,avg_toi_minutes,goals_per_60,assists_per_60,points_per_60,height_cm,weight_kg,base_similarity,same_hand,similarity
0,8479420,Tage Thompson,29,C,R,26,81,40,41,81,...,1.000000,19.243210,1.54,1.58,3.12,198,100,97.1,0,95.1
1,8478401,Pavel Zacha,29,C,L,6,78,30,37,67,...,0.858974,16.846154,1.37,1.69,3.06,193,96,94.0,1,94.2
2,8477500,Bo Horvat,31,C,L,9,68,31,27,58,...,0.852941,20.763235,1.32,1.15,2.46,185,102,94.0,1,94.1
3,8477946,Dylan Larkin,30,C,L,15,74,34,33,67,...,0.905405,20.186486,1.37,1.33,2.69,185,93,93.9,1,94.0
4,8480014,Gabriel Vilardi,27,C,R,11,82,30,39,69,...,0.841463,18.740244,1.17,1.52,2.69,191,98,95.9,0,94.0
5,8479987,Morgan Geekie,28,C,R,67,81,41,29,70,...,0.864198,17.406173,1.74,1.23,2.98,191,96,95.9,0,94.0
6,8476459,Mika Zibanejad,33,C,R,6,81,35,45,80,...,0.987654,20.897531,1.24,1.60,2.84,188,94,92.4,0,90.5
7,8478493,Joel Eriksson Ek,29,C,L,20,70,19,32,51,...,0.728571,19.078571,0.85,1.44,2.29,191,94,90.3,1,90.5
8,8481596,Shane Pinto,26,C,R,32,72,23,24,47,...,0.652778,18.712500,1.02,1.07,2.09,191,93,92.1,0,90.2
9,8475754,Brock Nelson,35,C,L,30,81,33,32,65,...,0.802469,19.653086,1.24,1.21,2.45,193,93,89.1,1,89.3


In [47]:
display(find_comparables_v2(8480800, 10))

,playerId,player,age,position,shoots_catches,draftPick,games,goals,assists,points,...,points_per_game,avg_toi_minutes,goals_per_60,assists_per_60,points_per_60,height_cm,weight_kg,base_similarity,same_hand,similarity
0,8483457,Lane Hutson,22,D,L,62,82,13,72,85,...,1.036585,23.771951,0.40,2.22,2.62,175,73,93.6,1,93.9
1,8479323,Adam Fox,28,D,R,66,55,9,46,55,...,1.000000,23.629091,0.42,2.12,2.54,180,84,97.6,0,92.7
2,8480036,Miro Heiskanen,27,D,L,3,77,9,55,64,...,0.831169,25.472727,0.28,1.68,1.96,188,89,92.2,1,92.6
3,8477504,Josh Morrissey,31,D,L,13,77,14,41,55,...,0.714286,24.720779,0.44,1.29,1.73,183,88,91.7,1,92.1
4,8478407,Vince Dunn,30,D,L,56,81,11,34,45,...,0.555556,21.662963,0.38,1.16,1.54,183,91,89.2,1,89.8
5,8478469,Thomas Chabot,29,D,L,18,57,7,24,31,...,0.543860,22.578947,0.33,1.12,1.45,185,91,87.7,1,88.4
6,8480069,Cale Makar,28,D,R,4,75,20,59,79,...,1.053333,24.848000,0.64,1.90,2.54,183,85,91.9,0,87.3
7,8479425,Filip Hronek,29,D,R,53,82,8,41,49,...,0.597561,25.006098,0.23,1.20,1.43,183,86,90.7,0,86.1
8,8476906,Shayne Gostisbehere,33,D,L,78,55,13,37,50,...,0.909091,19.232727,0.74,2.10,2.84,180,83,82.3,1,83.2
9,8479325,Charlie McAvoy,29,D,R,14,69,11,53,64,...,0.927536,24.389855,0.39,1.89,2.28,185,96,87.2,0,82.9


In [48]:
display(find_comparables_v2(8480069, 10))

,playerId,player,age,position,shoots_catches,draftPick,games,goals,assists,points,...,points_per_game,avg_toi_minutes,goals_per_60,assists_per_60,points_per_60,height_cm,weight_kg,base_similarity,same_hand,similarity
0,8479323,Adam Fox,28,D,R,66,55,9,46,55,...,1.000000,23.629091,0.420000,2.120000,2.540000,180,84,96.7,1,96.9
1,8480803,Evan Bouchard,27,D,R,10,82,21,75,96,...,1.170732,24.678049,0.620000,2.220000,2.850000,191,87,95.0,1,95.3
2,8478178,Darren Raddysh,30,D,R,<NA>,73,24,48,72,...,0.986301,22.691781,0.870000,1.740000,2.610000,185,92,93.6,1,93.9
3,8474578,Erik Karlsson,36,D,R,15,75,18,56,74,...,0.986667,23.600000,0.610000,1.900000,2.510000,183,84,90.1,1,90.6
4,8479325,Charlie McAvoy,29,D,R,14,69,11,53,64,...,0.927536,24.389855,0.390000,1.890000,2.280000,185,96,89.5,1,90.0
5,8480800,Quinn Hughes,27,D,L,7,74,7,69,76,...,1.027027,27.737838,0.202629,2.017983,2.220612,178,82,91.9,0,87.3
6,8476906,Shayne Gostisbehere,33,D,L,78,55,13,37,50,...,0.909091,19.232727,0.740000,2.100000,2.840000,180,83,90.7,0,86.1
7,8478460,Zach Werenski,29,D,L,8,75,24,60,84,...,1.120000,26.614667,0.720000,1.800000,2.520000,188,97,90.0,0,85.5
8,8477504,Josh Morrissey,31,D,L,13,77,14,41,55,...,0.714286,24.720779,0.440000,1.290000,1.730000,183,88,89.6,0,85.2
9,8480839,Rasmus Dahlin,26,D,L,1,77,19,55,74,...,0.961039,24.185714,0.610000,1.770000,2.380000,191,93,89.6,0,85.1
